In [ ]:
import os
import h5py
from matplotlib import pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

In [ ]:
folder = "/home/mvecsei/NN_tests"
file_list = [file for file in os.listdir(folder) if "withrecon" in file]
# exp_id_list = [file.split("_")[0] for file in os.listdir("/home/apdcam/repos/flap_modules/flap_spade/recon") if "withrecon" in file]

In [ ]:
for file in file_list:
    filename = os.path.join(folder, file)
    with h5py.File(filename, mode='r') as f:
        r=f['grid'][()]
        downsampling=1
        time_instances=[float(i[14:19]) for i in f["tags"][()]]
        light=f['emission'][()]
        light_normalized=light/np.array([np.trapz(l,x=r) for l in light])[:,None]*(-1)
        light_recon=f['light_recon'][()]
        light_recon_r=f['light_recon_r'][()]
        light_recon_normalized=light_recon/np.array([np.trapz(l,x=light_recon_r) for l in light_recon])[:,None]*(-1)

        # allow slight out-of-bounds by extrapolating
        light_recon_interpolated = np.stack([
            interp1d(light_recon_r, l, kind='linear', bounds_error=False, fill_value='extrapolate')(r)
            for l in light_recon_normalized
        ], axis=0)
        reldiff=(light_normalized-light_recon_interpolated)/light_normalized
        # interpolate nans and np.float64(inf) with linear interpolation
        for i in range(reldiff.shape[0]):
            nans = np.isnan(reldiff[i])
            infs = np.isinf(reldiff[i])
            nans = nans | infs
            if np.any(nans) or np.any(infs):
                valid = ~(nans | infs)
                if np.any(valid):
                    reldiff[i, nans] = np.interp(
                        np.flatnonzero(nans),
                        np.flatnonzero(valid),
                        reldiff[i, valid]
                    )
                else:
                    reldiff[i, nans] = 0.0
        plt.figure(figsize=(12,10))
    
        plt.subplot(3,1,1)
        im = plt.pcolormesh(time_instances[::downsampling], r, light_normalized[::downsampling].T, cmap='RdBu_r')
        cbar = plt.colorbar(im)
        cbar.ax.set_ylabel('mV',fontsize=18, labelpad=10)
        cbar.ax.tick_params(labelsize=12)
        vmax = max(np.max(light_normalized), np.max(light_recon_normalized))
        im.set_clim(0, 20)
        plt.title('Light Profiles '+str(f['ID'][()]),fontsize=20)
        plt.xlabel('Time',fontsize=18, labelpad=10)
        plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
        plt.xticks(fontsize=14)
        plt.yticks(fontsize=14)
    
        plt.subplot(3,1,2)
        im = plt.pcolormesh(time_instances[::downsampling], light_recon_r, light_recon_normalized[::downsampling].T, cmap='RdBu_r')
        cbar = plt.colorbar(im)
        cbar.ax.set_ylabel('mV',fontsize=18, labelpad=10)
        cbar.ax.tick_params(labelsize=12)
        vmax = max(np.max(light_normalized), np.max(light_recon_normalized))
        im.set_clim(0, 20)
        plt.title('Reconstructed Light Profiles '+str(f['ID'][()]),fontsize=20)
        plt.xlabel('Time',fontsize=18, labelpad=10)
        plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
        plt.xticks(fontsize=14)
        plt.yticks(fontsize=14)
    
        plt.subplot(3,1,3)
        im = plt.pcolormesh(time_instances[::downsampling], r, 100*(reldiff)[::downsampling].T, cmap='RdBu_r')
        cbar = plt.colorbar(im)
        cbar.ax.set_ylabel('%',fontsize=18, labelpad=10)
        cbar.ax.tick_params(labelsize=12)
        vmax = np.mean(reldiff)
        im.set_clim(-50, 50)
        plt.title('Relative difference in percents '+str(f['ID'][()]),fontsize=20)
        plt.xlabel('Time',fontsize=18, labelpad=10)
        plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
        plt.xticks(fontsize=14)
        plt.yticks(fontsize=14)
        plt.tight_layout()

        plt.figure(figsize=(12,10))
        plt.hist(100*(reldiff)[10:-10,10:-10].flatten(), bins=40, range=(-100, 100))


In [ ]:
r

In [ ]:
plt.figure(figsize=(15,4))
im=plt.pcolormesh(time_instances, light_recon_r, light_recon.T, cmap='RdBu_r')
im.set_clim(0, 0.1)
plt.show

In [ ]:
plt.figure(figsize=(15,4))
im=plt.pcolormesh(time_instances, r, light.T, cmap='RdBu_r')
im.set_clim(0, 205)
plt.show

In [ ]:
light_recon_normalized

In [ ]:
import h5py
with h5py.File(os.path.join("/data2/W7-X/processed_data/APDCAM/NN_recon/","Dataset_Na_0_we_20250520.010.h5"), 'r') as f:
    print(f.keys())
    pred=f['density'][()]
    r=f['grid'][()]
    time_instances=[float(i[14:19]) for i in f["tags"][()]]
plt.figure(figsize=(15,4))
plt.pcolormesh(time_instances, r, pred.T, cmap='RdBu_r')

In [ ]:
with h5py.File(os.path.join("/data2/W7-X/processed_data/APDCAM/NN_recon/","Dataset_Na_0_we_20250513.032.h5"), 'r') as f:
    print(f.keys())
    pred=f['density'][()]
    r=f['grid'][()]
    time_instances=[float(i[14:19]) for i in f["tags"][()]]
plt.figure(figsize=(15,4))
im = plt.pcolormesh(time_instances[::10],r, pred[::10].T*1e19, cmap='RdBu_r')
cbar = plt.colorbar(im)
cbar.ax.set_ylabel('$m^{-3}$',fontsize=18, labelpad=10)
cbar.ax.tick_params(labelsize=16)
vmax = np.max(4*1e19)
im.set_clim(0, vmax)
plt.title('Predicted Density Profiles '+"for shot 20250513.032",fontsize=20)
plt.xlabel('Time (s)',fontsize=18, labelpad=10)
plt.ylabel('Major radius (m)',fontsize=18, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.savefig("time_evolution_shot_"+"20250513032"+".png", bbox_inches='tight')
plt.show()
